# Importing an `snnTorch` SNN model to LAVA

In [7]:
# Show current directory
import os
curr_dir = os.getcwd()
print(curr_dir)

# Check if the current WD is the file location
if "/lava-dl/src/nir" not in os.getcwd():
    # Set working directory to this file location
    file_location = f"{os.getcwd()}/lava-dl/src/nir"
    print("File Location: ", file_location)

    # Change the current working Directory
    os.chdir(file_location)

    # New Working Directory
    print("New Working Directory: ", os.getcwd())

/home/monkin/Desktop/feup/thesis/lava-dl/src/nir


In [8]:
import numpy as np
import nir
import matplotlib.pyplot as plt

nir_network = nir.read("nir_model_cuba.nir")

In [9]:
# Print the network summary
print(f"NIR Network Info: Nº Nodes: {len(nir_network.nodes)} | Nº Edges: {len(nir_network.edges)}")

NIR Network Info: Nº Nodes: 6 | Nº Edges: 5


In [12]:
from nir_to_lava import ImportConfig, LavaLibrary, import_from_nir

config = ImportConfig(
    dt=1e-4, fixed_pt=False, on_chip=False, library_preference=LavaLibrary.LavaDl,
)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/monkin/Desktop/feup/thesis/.venv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/monkin/Desktop/feup/thesis/.venv/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [10]:
nir2lava_network = import_from_nir(nir_network, config)

node 0: Affine
node 1: CubaLIF
node 2: Affine
node 3: CubaLIF
node output: Output


In [11]:
print(nir2lava_network)

NIR2LavaDLNetwork(
  (blocks): ModuleList(
    (0): Dense(784, 500, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    (1): Dense(
      (neuron): Neuron()
      (synapse): Dense(7, 7, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    )
    (2): Dense(500, 10, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    (3): Dense(
      (neuron): Neuron()
      (synapse): Dense(7, 7, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    )
  )
)


Under a certain set of parameters, I was able to define a network in `snnTorch`, export it to the `NIR` format and import it to `Lava`. This network used CUBA LIF neurons, `snn.Synaptic` (in snn-torch) or `LIF` (in lava).

## Import NIR Network created manually

In [54]:
MANUAL_CREATION = True

ng = None
dt = 1e-4
if MANUAL_CREATION:
    # TODO: This is not the original code. I changed it because it had errors..
    """ ng = nir.NIRGraph(
        nodes={
            "input": nir.Input(input_type=np.array([3])),
            # "affine": nir.Affine(weight=np.array([[8, 2, 10], [14, 3, 14]]).T, bias=np.array([1, 2])),
            "lif": nir.CubaLIF(
                # tau=np.array([1] * 2),
                r=np.array([1 / 1e-4] * 2),
                tau_syn=np.array([1] * 2),
                tau_mem=np.array([1] * 2),
                v_leak=np.array([0] * 2),
                v_threshold=np.array([1] * 2),
            ),
            "out": nir.Output(np.array([3])),
        },
        edges=[
            # ("input", "affine"),
            # ("affine", "lif"),
            ("input", "lif"),
            ("lif", "out")
        ],
    ) """
    ng= nir.NIRGraph(
        nodes={
            "input": nir.Input(input_type=np.array([3])),
            "affine": nir.Affine(weight=np.array(
                [[8, 2, 10], [14, 3, 14]], dtype=float).T, bias=np.array([0, 0, 0], dtype=float)),
            "lif": nir.CubaLIF(
                tau_syn=np.array([0.3] * 2),
                tau_mem=np.array([0.2] * 2),
                r=np.array([0.2/dt] * 2),       # Have to define dt but its value is fixed to: tau_mem/dt
                v_leak=np.array([0] * 2),
                v_threshold=np.array([1] * 2),
            ),
            "output": nir.Output(np.array([3])),   # nir.Output(input_type=np.array([3])),
        },
        edges=[("input", "affine"), ("affine", "lif"), ("lif", "output")],
    )

In [55]:
config_manual = ImportConfig(
    dt=1e-4, fixed_pt=False, on_chip=False, library_preference=LavaLibrary.LavaDl,
)

In [56]:
# Print the manually created network
print("NG: ", ng.nodes.keys(), ng.edges)

NG:  dict_keys(['input', 'affine', 'lif', 'output']) [('input', 'affine'), ('affine', 'lif'), ('lif', 'output')]


In [60]:
nir2lava_manual_network = import_from_nir(ng, config_manual)

node affine: Affine
node lif: CubaLIF
[warning] scaling weights according to w_in -> w_scale=0.0003333333333333334
node output: Output


In [61]:
print(nir2lava_manual_network)

NIR2LavaDLNetwork(
  (blocks): ModuleList(
    (0): Dense(2, 3, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    (1): Dense(
      (neuron): Neuron()
      (synapse): Dense(7, 7, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    )
  )
)
